# Diagnóstico Geoespacial de Desmatamento Oficial (DETER / INPE) — PES Charapucu
### Núcleo de Geotecnologias (NGEO) / IDEFLOR-Bio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/restaura-pes-charapucu/blob/main/notebooks/dashboard_charapucu.ipynb)

Este notebook orquestra o **Diagnóstico Geoespacial de Desmatamento Oficial** auditado pelo **DETER (INPE / TerraBrasilis)** no interior do **Parque Estadual Charapucu** (Afuá / Arquipélago do Marajó - PA).

**Destaques do Diagnóstico Oficial:**
* **Fonte Primária Oficial:** DETER - Amazônia Legal (INPE / MMA / TerraBrasilis)
* **Classe Oficial:** `DESMATAMENTO_CR` (Desmatamento por Corte Raso)
* **Data da Detecção pelo INPE:** 20/07/2019 (Sensor AWFI / Satélite CBERS-4)
* **Área Total Auditada:** **$183{,}76\text{ ha}$** (2 glebas contíguas — **141,4%** da meta de $130\text{ ha}$)
* **Status Ecológico Atual:** Várzea em processo acelerado de **regeneração natural secundária / capoeira**, alvo prioritário para planejamento de campo da Gestão do PES Charapucu e da DGBIO.

---

In [ ]:
# [1] Instalação de dependências e configuração de ambiente (Google Colab / Local)
import os
import sys

try:
    import google.colab
    IN_COLAB = True
    print('Ambiente Google Colab detectado.')
    !pip install -q geopandas folium shapely
    
    # Se aberto diretamente no Colab pelo GitHub, clona o repositório para ter os dados
    if not os.path.exists('outputs') and not os.path.exists('data'):
        if not os.path.exists('restaura-pes-charapucu'):
            print('Clonando repositório para o ambiente de execução...')
            !git clone https://github.com/NGEO-IDEFLOR-Bio/restaura-pes-charapucu.git
        if os.path.exists('restaura-pes-charapucu'):
            %cd restaura-pes-charapucu
            print('Diretório de trabalho ajustado para:', os.getcwd())
except ImportError:
    IN_COLAB = False
    print('Ambiente Local / Jupyter detectado.')

print('Ambiente pronto!')

In [ ]:
# [2] Carga dos Dados Geoespaciais do DETER/INPE e Limite da UC
from pathlib import Path
import geopandas as gpd
import pandas as pd

caminhos_deter = [
    Path('outputs/vetores/banco_restauracao_pes_charapucu.geojson'),
    Path('data/processed/banco_restauracao_consolidado.geojson'),
    Path('../outputs/vetores/banco_restauracao_pes_charapucu.geojson'),
    Path('data/raw/deter_alertas_charapucu.geojson')
]

arquivo_dados = next((p for p in caminhos_deter if p.exists()), None)
if arquivo_dados is None:
    raise FileNotFoundError('Base vetorial não localizada. Execute o script de consolidação.')

print(f'Carregando dados DETER de: {arquivo_dados}...')
gdf_deter = gpd.read_file(arquivo_dados).to_crs('EPSG:4326')

# Converter campos de data/Timestamp para string para serialização do Folium
for c in gdf_deter.columns:
    if str(gdf_deter[c].dtype).startswith('datetime') or c == 'data_alerta':
        gdf_deter[c] = gdf_deter[c].astype(str)

caminhos_limite = [
    Path('data/raw/limite_pes_charapucu.geojson'),
    Path('../data/raw/limite_pes_charapucu.geojson')
]
arquivo_limite = next((p for p in caminhos_limite if p.exists()), None)
gdf_parque = gpd.read_file(arquivo_limite).to_crs('EPSG:4326') if arquivo_limite else None

print(f'✅ {len(gdf_deter)} glebas oficiais carregadas! Área total: {gdf_deter["area_ha"].sum():.2f} ha (141.4% da meta de 130 ha)')

In [ ]:
# [3] Mapa Navegável com Basemaps Esri, Legenda Flutuante e Botão para Nova Guia
from pathlib import Path
import folium
from folium import GeoJson, GeoJsonTooltip, GeoJsonPopup
from branca.element import Element
from IPython.display import display, HTML

# Centroide dos alertas DETER (-50.398, -0.395)
centro_lat, centro_lon = -0.3950, -50.3980

m = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=13,
    tiles=None,
    control_scale=True
)

# Basemaps Esri Livres (Sem exigência de chave e sem bloqueio)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
    name='Satélite (Esri World Imagery)',
    overlay=False,
    control=True
).add_to(m)

folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Light Gray',
    name='Mapa Claro (Esri Light Gray)',
    overlay=False,
    control=True
).add_to(m)

folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Topo Map',
    name='Topográfico (Esri Topo)',
    overlay=False,
    control=True
).add_to(m)

# Camada do Limite Oficial do Parque
if gdf_parque is not None and not gdf_parque.empty:
    folium.GeoJson(
        gdf_parque,
        name='Limite Oficial PES Charapucu',
        style_function=lambda x: {
            'color': '#00ffff',
            'weight': 2.5,
            'fillOpacity': 0.0,
            'dashArray': '4, 4'
        },
        tooltip='<b>Parque Estadual Charapucu</b> (72.506 ha)'
    ).add_to(m)

# Camada dos Alertas DETER/INPE
def estilo_deter(f):
    return {
        'color': '#c0392b',
        'weight': 2.5,
        'fillColor': '#e74c3c',
        'fillOpacity': 0.55
    }

campos_tip = ['id_gleba', 'classe_inpe', 'data_alerta', 'area_ha', 'dist_margem_m']
aliases_tip = ['Gleba:', 'Classe INPE:', 'Data Alerta:', 'Área (ha):', 'Dist. Rio (m):']
campos_tip = [c for c in campos_tip if c in gdf_deter.columns]
aliases_tip = aliases_tip[:len(campos_tip)]

campos_pop = ['id_gleba', 'id_inpe', 'fonte', 'classe_inpe', 'data_alerta', 'diagnostico', 'area_ha', 'classe_area', 'acesso_fluvial', 'dist_margem_m', 'setor_fluvial']
aliases_pop = ['ID Gleba:', 'ID Alerta INPE:', 'Fonte Oficial:', 'Classe DETER:', 'Data Detecção:', 'Diagnóstico:', 'Área Calculada:', 'Porte:', 'Acesso Fluvial:', 'Distância à Margem:', 'Setor:']
campos_pop = [c for c in campos_pop if c in gdf_deter.columns]
aliases_pop = aliases_pop[:len(campos_pop)]

GeoJson(
    gdf_deter,
    name='Alertas de Corte Raso (DETER/INPE - 183.76 ha)',
    style_function=estilo_deter,
    tooltip=GeoJsonTooltip(fields=campos_tip, aliases=aliases_tip, localize=True),
    popup=GeoJsonPopup(fields=campos_pop, aliases=aliases_pop, localize=True)
).add_to(m)

# Legenda Flutuante
legenda_html = '''
<div style="position: fixed; bottom: 25px; left: 25px; width: 260px; background-color: rgba(255, 255, 255, 0.95); border: 2px solid #bdc3c7; border-radius: 8px; z-index: 9999; font-size: 11px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; padding: 12px 14px; box-shadow: 0 2px 8px rgba(0,0,0,0.2);">
    <div style="font-weight: 700; margin-bottom: 6px; font-size: 12px; color: #2c3e50; border-bottom: 1px solid #e1e8ed; padding-bottom: 4px;">Diagnóstico Oficial — DETER/INPE</div>
    <div style="display: flex; align-items: center; margin-bottom: 6px;">
        <span style="background: #e74c3c; width: 14px; height: 14px; display: inline-block; border-radius: 3px; margin-right: 8px; border: 1px solid #c0392b;"></span>
        <span style="color: #2c3e50;"><b>Corte Raso Oficial</b> (183,76 ha)</span>
    </div>
    <div style="font-size: 10px; color: #7f8c8d; margin-left: 22px; margin-bottom: 6px;">Detectado pelo INPE em 20/07/2019</div>
    <div style="display: flex; align-items: center; margin-top: 6px; padding-top: 6px; border-top: 1px dashed #d5dbdb;">
        <span style="border-top: 2px dashed #00ffff; width: 18px; display: inline-block; margin-right: 8px;"></span>
        <span style="color: #555; font-size: 10px;">Limite Oficial PES Charapucu</span>
    </div>
</div>
'''
m.get_root().html.add_child(Element(legenda_html))
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# Salvar mapa local em HTML
Path('outputs').mkdir(parents=True, exist_ok=True)
caminho_html = 'outputs/mapa_deter_charapucu.html'
m.save(caminho_html)

# Botão Interativo para Abrir em Nova Guia
html_botao = f'''
<div style="margin: 15px 0; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif;">
    <a href="{caminho_html}" target="_blank" style="
        display: inline-flex;
        align-items: center;
        gap: 8px;
        background-color: #27ae60;
        color: #ffffff;
        font-size: 13px;
        font-weight: 700;
        padding: 10px 18px;
        border-radius: 6px;
        text-decoration: none;
        box-shadow: 0 2px 6px rgba(39, 174, 96, 0.35);
        transition: all 0.2s ease;
    " onmouseover="this.style.backgroundColor='#219150'" onmouseout="this.style.backgroundColor='#27ae60'">
        <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><path d="M18 13v6a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h6"></path><polyline points="15 3 21 3 21 9"></polyline><line x1="10" y1="14" x2="21" y2="3"></line></svg>
        Abrir Mapa em Nova Guia (Tela Cheia)
    </a>
    <span style="margin-left: 12px; font-size: 12px; color: #7f8c8d;">Arquivo exportado: <code>{caminho_html}</code></span>
</div>
'''

display(HTML(html_botao))
display(m)